# SQL Query Fundamentals (SELECT, WHERE, GROUP BY)

In the last lesson, we built the structure of our database using Data Definition Language (DDL). Now, it is time to actually put data inside and learn how to ask the database questions.

The commands we use to fetch and analyze data are part of **Data Query Language (DQL)**. For a Data Scientist or Data Analyst, writing queries is the most important database skill you will learn. It is how you extract the exact subset of data you need to feed into your machine learning models or dashboards.

Let's set up our temporary SQLite database, insert some realistic data, and start querying!

In [1]:
import sqlite3
import pandas as pd

# 1. Connect to an in-memory database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Create a Sales table and insert some dummy data
cursor.executescript("""
CREATE TABLE Sales (
    transaction_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    category TEXT,
    amount DECIMAL(10, 2),
    purchase_date DATE
);

INSERT INTO Sales (customer_name, category, amount, purchase_date) VALUES 
('Alice', 'Electronics', 1200.50, '2023-10-01'),
('Bob', 'Clothing', 45.00, '2023-10-02'),
('Charlie', 'Electronics', 850.00, '2023-10-02'),
('Alice', 'Clothing', 30.00, '2023-10-03'),
('Eve', 'Home', 150.75, '2023-10-03'),
('Bob', 'Electronics', 999.99, '2023-10-04'),
('Alice', 'Home', 45.00, '2023-10-05');
""")
print("✅ Database ready and populated with Sales data!")

✅ Database ready and populated with Sales data!


# 1. SELECT: Choosing Your Columns
The `SELECT` statement is the foundation of every query. It tells the database exactly *which columns* you want to see.

* **`SELECT *`**: The asterisk means "Give me ALL the columns."
* **`SELECT column1, column2`**: Gives you only the specific columns you ask for.

In [2]:
# Query 1: Give me everything in the table
query_all = "SELECT * FROM Sales;"
print("--- All Data (SELECT *) ---")
display(pd.read_sql_query(query_all, conn))

# Query 2: I only care about who bought what, not the price or date
query_specific = "SELECT customer_name, category FROM Sales;"
print("\n--- Specific Columns ---")
display(pd.read_sql_query(query_specific, conn))

--- All Data (SELECT *) ---


,transaction_id,customer_name,category,amount,purchase_date
0,1,Alice,Electronics,1200.50,2023-10-01
1,2,Bob,Clothing,45.00,2023-10-02
2,3,Charlie,Electronics,850.00,2023-10-02
3,4,Alice,Clothing,30.00,2023-10-03
4,5,Eve,Home,150.75,2023-10-03
5,6,Bob,Electronics,999.99,2023-10-04
6,7,Alice,Home,45.00,2023-10-05



--- Specific Columns ---


,customer_name,category
0,Alice,Electronics
1,Bob,Clothing
2,Charlie,Electronics
3,Alice,Clothing
4,Eve,Home
5,Bob,Electronics
6,Alice,Home


# 2. WHERE: Filtering Your Rows
Usually, you don't want to look at millions of rows of data. The `WHERE` clause allows you to filter the data based on specific conditions.

* **Math Operators**: `=`, `>`, `<`, `>=`, `<=`, `!=` (not equal).
* **Logic Operators**: `AND`, `OR`.
* **Pattern Matching**: `LIKE` (e.g., `LIKE 'A%'` finds anything starting with 'A').

In [3]:
# Find all purchases over $500
query_expensive = """
SELECT * FROM Sales 
WHERE amount > 500;
"""
print("--- Expensive Purchases (WHERE) ---")
display(pd.read_sql_query(query_expensive, conn))

# Find Electronics bought by Alice
query_alice_tech = """
SELECT * FROM Sales 
WHERE customer_name = 'Alice' AND category = 'Electronics';
"""
print("\n--- Alice's Electronics (WHERE + AND) ---")
display(pd.read_sql_query(query_alice_tech, conn))

--- Expensive Purchases (WHERE) ---


,transaction_id,customer_name,category,amount,purchase_date
0,1,Alice,Electronics,1200.50,2023-10-01
1,3,Charlie,Electronics,850.00,2023-10-02
2,6,Bob,Electronics,999.99,2023-10-04



--- Alice's Electronics (WHERE + AND) ---


,transaction_id,customer_name,category,amount,purchase_date
0,1,Alice,Electronics,1200.5,2023-10-01


# 3. ORDER BY: Sorting the Results
By default, a database returns data in whatever order it was saved. To sort your results alphabetically or numerically, use `ORDER BY`.

* **`ASC`**: Ascending order (A-Z, 1-100). This is the default.
* **`DESC`**: Descending order (Z-A, 100-1).

In [4]:
# Sort the sales by amount, from highest to lowest
query_sort = """
SELECT customer_name, amount 
FROM Sales 
ORDER BY amount DESC;
"""
print("--- Sorted by Amount (ORDER BY DESC) ---")
display(pd.read_sql_query(query_sort, conn))

--- Sorted by Amount (ORDER BY DESC) ---


,customer_name,amount
0,Alice,1200.50
1,Bob,999.99
2,Charlie,850.00
3,Eve,150.75
4,Bob,45.00
5,Alice,45.00
6,Alice,30.00


# 4. GROUP BY: Aggregating Data
As a Data Scientist, you often need to summarize data. `GROUP BY` squashes rows that share the same value in a specific column into a single row. You must use an **Aggregate Function** (like `SUM`, `COUNT`, `AVG`, `MAX`, `MIN`) to tell the database *how* to squash the numbers together.

In [5]:
# How much money did each customer spend in total?
query_group = """
SELECT customer_name, SUM(amount) as total_spent, COUNT(transaction_id) as number_of_items
FROM Sales
GROUP BY customer_name;
"""
print("--- Total Spend per Customer (GROUP BY + SUM) ---")
display(pd.read_sql_query(query_group, conn))

--- Total Spend per Customer (GROUP BY + SUM) ---


,customer_name,total_spent,number_of_items
0,Alice,1275.50,3
1,Bob,1044.99,2
2,Charlie,850.00,1
3,Eve,150.75,1


*(Notice how we used `as total_spent`. This is called an **Alias**, which renames the column in our final output to make it easier to read!)*

# 5. HAVING: Filtering Aggregated Groups
Here is a very common beginner mistake: You cannot use `WHERE` to filter on an aggregate function (like `SUM(amount)`). The `WHERE` clause filters rows *before* they are grouped. 

If you want to filter the groups *after* they are squashed together, you must use **`HAVING`**.

In [6]:
# Show me only the customers who have spent more than $1000 in total
query_having = """
SELECT customer_name, SUM(amount) as total_spent
FROM Sales
GROUP BY customer_name
HAVING SUM(amount) > 1000;
"""
print("--- High Rollers Only (HAVING) ---")
display(pd.read_sql_query(query_having, conn))

# Close the connection
conn.close()

--- High Rollers Only (HAVING) ---


,customer_name,total_spent
0,Alice,1275.50
1,Bob,1044.99


## Real-World Use Case or Analogy:
Think of writing an SQL Query like **Ordering at a highly customized Restaurant**:

* **`FROM`**: Choosing the restaurant you are eating at. (e.g., "We are eating at `Sales`").
* **`WHERE`**: Stating your dietary restrictions upfront. (e.g., "Only show me items `WHERE category = 'Vegetarian'`). The chef immediately throws away all the meat options before doing anything else.
* **`GROUP BY`**: Asking the chef to combine things. (e.g., "Take all the vegetarian dishes and group them by `Spiciness Level`").
* **`HAVING`**: Applying a rule to the combined dishes. (e.g., "Only show me the spiciness levels `HAVING` more than 3 dishes available").
* **`SELECT`**: Deciding exactly what you want the waiter to bring to your table. (e.g., "Bring me the `Name` of the dish and the `Price`").
* **`ORDER BY`**: Telling the waiter how to place them on the table. (e.g., "Put them in order from cheapest to most expensive").

---